In [ ]:


import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

SAMPLE_RATE = 44100
N_MFCC      = 40
N_MELS      = 128
N_FFT       = 2048
HOP_LENGTH  = 512

CSV_OUTPUT = 'metadata.csv'
df = pd.read_csv(CSV_OUTPUT)
print(f'Loaded {len(df)} files')


In [ ]:


def extract_features(file_path, sr=SAMPLE_RATE):
    """
    Extracts audio features from a single .wav file.

    Features extracted by Bharat:
      - MFCCs        (mean + std of 40 coefficients)   →  80 features
      - Delta MFCCs  (mean + std of 40 coefficients)   →  80 features
      - Delta2 MFCCs (mean + std of 40 coefficients)   →  80 features
      - Mel Spectrogram (mean + std of 128 bins)       → 256 features
      - Chroma STFT  (mean + std of 12 bins)           →  24 features
    ──────────────────────────────────────────────────────────
    Subtotal (Bharat)                                  → 520 features

    Additional spectral features added by Aasim       →  24 features
    Tonnetz features added by Khadija                  →  12 features
    ──────────────────────────────────────────────────────────
    GRAND TOTAL                                        → 556 features
    """
    try:
        
        audio, _ = librosa.load(file_path, sr=sr, mono=True)

        features = {}

       
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC,
                                     n_fft=N_FFT, hop_length=HOP_LENGTH)
        for i in range(N_MFCC):
            features[f'mfcc_{i+1}_mean'] = np.mean(mfcc[i])
            features[f'mfcc_{i+1}_std']  = np.std(mfcc[i])

        
        delta_mfcc = librosa.feature.delta(mfcc)
        for i in range(N_MFCC):
            features[f'delta_mfcc_{i+1}_mean'] = np.mean(delta_mfcc[i])
            features[f'delta_mfcc_{i+1}_std']  = np.std(delta_mfcc[i])

        
        delta2_mfcc = librosa.feature.delta(mfcc, order=2)
        for i in range(N_MFCC):
            features[f'delta2_mfcc_{i+1}_mean'] = np.mean(delta2_mfcc[i])
            features[f'delta2_mfcc_{i+1}_std']  = np.std(delta2_mfcc[i])

       
        mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS,
                                              n_fft=N_FFT, hop_length=HOP_LENGTH)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        for i in range(N_MELS):
            features[f'mel_{i+1}_mean'] = np.mean(mel_db[i])
            features[f'mel_{i+1}_std']  = np.std(mel_db[i])

        
        chroma = librosa.feature.chroma_stft(y=audio, sr=sr,
                                              n_fft=N_FFT, hop_length=HOP_LENGTH)
        for i in range(chroma.shape[0]):
            features[f'chroma_{i+1}_mean'] = np.mean(chroma[i])
            features[f'chroma_{i+1}_std']  = np.std(chroma[i])

        return features, audio  

    except Exception as e:
        print(f'\n  [ERROR] {os.path.basename(file_path)}: {e}')
        return None, None

print('extract_features() function defined!')
print('Features covered by Bharat: MFCCs + Delta + Delta2 + Mel + Chroma = 520 features')
